# Projeto 1 — Financial Planning & Scenario Analysis

## Electronics e Personal Care

### Objetivo
Transformar o forecast comercial corrigido em uma análise financeira de:

- receita bruta;
- receita após desconto;
- custo estimado;
- lucro bruto;
- margem bruta;
- comparação entre cenários;
- sensibilidade de desconto;
- ponto de equilíbrio entre volume adicional e perda de preço.

> **Limitação da base:** não existe uma coluna de custo do produto.  
> Por isso, o custo será tratado como **premissa de planejamento**, explicitamente configurável e separado dos dados observados.


## 1. Bibliotecas e leitura dos resultados da fase anterior


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

FORECAST_FILE = "commercial_forecast_scenarios_8_weeks.csv"
SUMMARY_FILE = "commercial_scenario_summary_8_weeks.csv"

forecast = pd.read_csv(
    FORECAST_FILE,
    parse_dates=["forecast_week"]
)

scenario_summary = pd.read_csv(SUMMARY_FILE)

display(forecast.head())
display(scenario_summary)


## 2. Premissas financeiras

Como o custo unitário não existe na base, criaremos uma tabela de premissas por categoria.

Os percentuais abaixo são **valores ilustrativos para modelagem** e devem ser ajustados quando houver dados reais.

A lógica é:

\[
Custo\ Unitário = Preço\ Base 	imes Percentual\ de\ Custo
\]

O custo é mantido constante entre cenários para permitir avaliar o impacto isolado do desconto.


In [ ]:
# Premissas editáveis
cost_assumptions = pd.DataFrame({
    "category": ["Electronics", "Personal Care"],
    "cost_pct_of_list_price": [0.65, 0.55]
})

display(cost_assumptions)


## 3. Construção das métricas financeiras

Para cada semana e cenário:

### Preço efetivo

\[
Preço\ Efetivo = Preço\ Base 	imes (1 - Desconto)
\]

### Receita líquida estimada

\[
Receita = Unidades\ Previstas 	imes Preço\ Efetivo
\]

### Custo estimado

\[
Custo = Unidades\ Previstas 	imes Custo\ Unitário
\]

### Lucro bruto

\[
Lucro\ Bruto = Receita - Custo
\]

### Margem bruta

\[
Margem = 
rac{Lucro\ Bruto}{Receita}
\]


In [ ]:
financial = forecast.merge(
    cost_assumptions,
    on="category",
    how="left"
)

financial["effective_price"] = (
    financial["assumed_price"] *
    (1 - financial["assumed_discount"] / 100)
)

financial["unit_cost_est"] = (
    financial["assumed_price"] *
    financial["cost_pct_of_list_price"]
)

financial["net_revenue_est"] = (
    financial["forecast_units"] *
    financial["effective_price"]
)

financial["cost_est"] = (
    financial["forecast_units"] *
    financial["unit_cost_est"]
)

financial["gross_profit_est"] = (
    financial["net_revenue_est"]
    - financial["cost_est"]
)

financial["gross_margin_pct"] = np.where(
    financial["net_revenue_est"] != 0,
    financial["gross_profit_est"]
    / financial["net_revenue_est"]
    * 100,
    np.nan
)

display(financial.head())


## 4. Resumo financeiro por cenário — 8 semanas


In [ ]:
financial_summary = (
    financial
    .groupby(["category", "scenario"], as_index=False)
    .agg(
        forecast_units=("forecast_units", "sum"),
        avg_list_price=("assumed_price", "mean"),
        avg_discount=("assumed_discount", "mean"),
        avg_effective_price=("effective_price", "mean"),
        net_revenue_est=("net_revenue_est", "sum"),
        cost_est=("cost_est", "sum"),
        gross_profit_est=("gross_profit_est", "sum")
    )
)

financial_summary["gross_margin_pct"] = (
    financial_summary["gross_profit_est"]
    / financial_summary["net_revenue_est"]
    * 100
)

display(financial_summary)


## 5. Comparação com o cenário-base

A pergunta financeira central é:

> **O ganho de volume obtido com mais desconto compensa a redução do preço efetivo?**

Comparamos receita, lucro e margem de cada cenário contra o cenário-base.


In [ ]:
base_financial = (
    financial_summary[
        financial_summary["scenario"] == "Base"
    ][[
        "category",
        "net_revenue_est",
        "gross_profit_est",
        "gross_margin_pct"
    ]]
    .rename(columns={
        "net_revenue_est": "base_revenue",
        "gross_profit_est": "base_profit",
        "gross_margin_pct": "base_margin"
    })
)

comparison = financial_summary.merge(
    base_financial,
    on="category",
    how="left"
)

comparison["revenue_change"] = (
    comparison["net_revenue_est"]
    - comparison["base_revenue"]
)

comparison["revenue_change_pct"] = (
    comparison["revenue_change"]
    / comparison["base_revenue"]
    * 100
)

comparison["profit_change"] = (
    comparison["gross_profit_est"]
    - comparison["base_profit"]
)

comparison["profit_change_pct"] = (
    comparison["profit_change"]
    / comparison["base_profit"]
    * 100
)

comparison["margin_change_pp"] = (
    comparison["gross_margin_pct"]
    - comparison["base_margin"]
)

display(
    comparison[[
        "category",
        "scenario",
        "forecast_units",
        "avg_discount",
        "net_revenue_est",
        "gross_profit_est",
        "gross_margin_pct",
        "revenue_change_pct",
        "profit_change_pct",
        "margin_change_pp"
    ]]
)


## 6. Visual — Receita por cenário


In [ ]:
for categoria in comparison["category"].unique():

    temp = comparison[
        comparison["category"] == categoria
    ].copy()

    plt.figure(figsize=(8, 4))

    bars = plt.bar(
        temp["scenario"],
        temp["net_revenue_est"]
    )

    plt.title(f"Receita estimada por cenário — {categoria}")
    plt.ylabel("Receita estimada")
    plt.xlabel("")

    plt.bar_label(
        bars,
        labels=[
            f"{v:,.0f}"
            for v in temp["net_revenue_est"]
        ],
        padding=4
    )

    plt.tight_layout()
    plt.show()


## 7. Visual — Lucro bruto por cenário


In [ ]:
for categoria in comparison["category"].unique():

    temp = comparison[
        comparison["category"] == categoria
    ].copy()

    plt.figure(figsize=(8, 4))

    bars = plt.bar(
        temp["scenario"],
        temp["gross_profit_est"]
    )

    plt.title(f"Lucro bruto estimado por cenário — {categoria}")
    plt.ylabel("Lucro bruto estimado")
    plt.xlabel("")

    plt.bar_label(
        bars,
        labels=[
            f"{v:,.0f}"
            for v in temp["gross_profit_est"]
        ],
        padding=4
    )

    plt.tight_layout()
    plt.show()


## 8. Visual — Margem bruta

Esse gráfico é importante porque um cenário pode aumentar o volume e até a receita, mas ainda deteriorar a margem.


In [ ]:
for categoria in comparison["category"].unique():

    temp = comparison[
        comparison["category"] == categoria
    ].copy()

    plt.figure(figsize=(8, 4))

    bars = plt.bar(
        temp["scenario"],
        temp["gross_margin_pct"]
    )

    plt.title(f"Margem bruta por cenário — {categoria}")
    plt.ylabel("Margem bruta (%)")
    plt.xlabel("")

    plt.bar_label(
        bars,
        labels=[
            f"{v:.1f}%"
            for v in temp["gross_margin_pct"]
        ],
        padding=4
    )

    plt.tight_layout()
    plt.show()


# 9. Break-even de volume para desconto

Agora calculamos quanto as unidades precisariam crescer para que um desconto maior **mantivesse o mesmo lucro bruto do cenário-base**.

Se:

\[
Lucro = Q 	imes (Preço\ Efetivo - Custo)
\]

então:

\[
Q_{break-even}
=

rac{Lucro_{Base}}
{Preço_{Efetivo,Novo} - Custo}
\]

Isso permite comparar:

- crescimento de unidades previsto pelo modelo;
- crescimento mínimo necessário para preservar o lucro.


In [ ]:
break_even_rows = []

for categoria in financial_summary["category"].unique():

    temp = financial_summary[
        financial_summary["category"] == categoria
    ].copy()

    base_row = temp[
        temp["scenario"] == "Base"
    ].iloc[0]

    cost_pct = cost_assumptions.loc[
        cost_assumptions["category"] == categoria,
        "cost_pct_of_list_price"
    ].iloc[0]

    list_price = base_row["avg_list_price"]
    unit_cost = list_price * cost_pct

    for _, row in temp.iterrows():

        unit_contribution = (
            row["avg_effective_price"]
            - unit_cost
        )

        if unit_contribution > 0:
            break_even_units = (
                base_row["gross_profit_est"]
                / unit_contribution
            )
        else:
            break_even_units = np.nan

        required_growth_pct = (
            break_even_units
            / base_row["forecast_units"]
            - 1
        ) * 100

        predicted_growth_pct = (
            row["forecast_units"]
            / base_row["forecast_units"]
            - 1
        ) * 100

        break_even_rows.append({
            "category": categoria,
            "scenario": row["scenario"],
            "predicted_units": row["forecast_units"],
            "predicted_growth_pct": predicted_growth_pct,
            "break_even_units": break_even_units,
            "required_growth_pct": required_growth_pct,
            "growth_gap_pp": (
                predicted_growth_pct
                - required_growth_pct
            )
        })

break_even = pd.DataFrame(break_even_rows)

display(break_even)


### Como interpretar `growth_gap_pp`

- valor **positivo**: o crescimento previsto supera o mínimo necessário para preservar o lucro;
- valor **negativo**: o aumento previsto de volume não compensa o desconto;
- valor próximo de zero: cenário aproximadamente indiferente em termos de lucro bruto.


# 10. Sensibilidade ao custo

Como o custo é uma premissa, não devemos depender de apenas um valor.

A seguir testamos diferentes estruturas de custo:

- 45%
- 55%
- 65%
- 75%

do preço-base.

Isso permite verificar se a decisão sobre desconto muda conforme a margem estrutural do produto.


In [ ]:
cost_sensitivity = []

cost_levels = [0.45, 0.55, 0.65, 0.75]

for categoria in financial_summary["category"].unique():

    cat_data = financial_summary[
        financial_summary["category"] == categoria
    ]

    for cost_pct in cost_levels:

        for _, row in cat_data.iterrows():

            unit_cost = (
                row["avg_list_price"]
                * cost_pct
            )

            revenue = (
                row["forecast_units"]
                * row["avg_effective_price"]
            )

            cost = (
                row["forecast_units"]
                * unit_cost
            )

            profit = revenue - cost

            margin = (
                profit / revenue * 100
                if revenue != 0
                else np.nan
            )

            cost_sensitivity.append({
                "category": categoria,
                "scenario": row["scenario"],
                "cost_pct_of_list_price": cost_pct * 100,
                "forecast_units": row["forecast_units"],
                "revenue": revenue,
                "profit": profit,
                "margin_pct": margin
            })

cost_sensitivity_df = pd.DataFrame(
    cost_sensitivity
)

display(cost_sensitivity_df.head(20))


## 11. Melhor cenário por categoria e estrutura de custo

Agora identificamos qual cenário gera o maior lucro estimado em cada nível de custo.


In [ ]:
best_scenario_by_cost = (
    cost_sensitivity_df
    .sort_values(
        ["category", "cost_pct_of_list_price", "profit"],
        ascending=[True, True, False]
    )
    .groupby(
        ["category", "cost_pct_of_list_price"],
        as_index=False
    )
    .first()
)

display(
    best_scenario_by_cost[[
        "category",
        "cost_pct_of_list_price",
        "scenario",
        "forecast_units",
        "revenue",
        "profit",
        "margin_pct"
    ]]
)


# 12. Conclusão desta fase

Esta análise separa dois efeitos:

### Efeito comercial
O modelo estima como a quantidade vendida pode responder a diferentes níveis de desconto.

### Efeito financeiro
O planejamento calcula se essa resposta de volume é suficiente para aumentar:

- receita;
- lucro;
- margem.

Um cenário que gera mais unidades **não é automaticamente melhor**.

A decisão correta depende da relação entre:

\[
Volume 	imes Preço 	imes Margem
\]

Esta etapa prepara diretamente o próximo projeto:

> **Pricing & Margin Optimization**

onde o desconto/preço deixa de ser apenas um cenário e passa a ser uma variável de decisão a ser otimizada.


## 13. Exportação


In [ ]:
financial.to_csv(
    "financial_forecast_weekly.csv",
    index=False,
    encoding="utf-8-sig"
)

financial_summary.to_csv(
    "financial_scenario_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison.to_csv(
    "financial_scenario_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

break_even.to_csv(
    "discount_break_even_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

cost_sensitivity_df.to_csv(
    "cost_sensitivity_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

best_scenario_by_cost.to_csv(
    "best_scenario_by_cost.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos gerados:")
print("- financial_forecast_weekly.csv")
print("- financial_scenario_summary.csv")
print("- financial_scenario_comparison.csv")
print("- discount_break_even_analysis.csv")
print("- cost_sensitivity_analysis.csv")
print("- best_scenario_by_cost.csv")
